In [1]:
!pip install -q -U albumentations ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00


In [1]:
import os
import cv2
import math
import pandas as pd
import albumentations as A

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
ORIG_BASE = "/content/drive/MyDrive/zeon_sys_dataset"
CSV_PATH = os.path.join(ORIG_BASE, "annotations.csv")
IMG_DIR = os.path.join(ORIG_BASE, "images")

OUT_DIR = "/content/tube_dataset"
os.makedirs(f"{OUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUT_DIR}/labels/train", exist_ok=True)

# ==========================================
# 2. ALBUMENTATIONS SETUP
# ==========================================
transform = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=45, p=0.8, border_mode=cv2.BORDER_CONSTANT, value=(0,0,0)),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
   bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']))

# ==========================================
# 3. PROCESSING LOOP
# ==========================================
df = pd.read_csv(CSV_PATH)
unique_images = df['image'].unique()
AUG_COPIES = 5  # Generates 1 original + 4 augmented copies

print(f"Processing {len(unique_images)} images into {len(unique_images) * AUG_COPIES} total samples...")

for img_name in unique_images:
    img_path = os.path.join(IMG_DIR, img_name)
    img = cv2.imread(img_path)
    if img is None: continue

    img_h, img_w = img.shape[:2]
    img_rows = df[df['image'] == img_name]

    # Extract original targets
    orig_bboxes = []
    orig_keypoints = []
    orig_cats = []

    for _, row in img_rows.iterrows():
        cx, cy = row['center_x'], row['center_y']
        angle_rad = math.radians(row['angle_deg'])

        R = 40.0
        jx = cx - R * math.cos(angle_rad)
        jy = cy + R * math.sin(angle_rad)
        tx = cx + R * math.cos(angle_rad)
        ty = cy - R * math.sin(angle_rad)

        # Store all 3 keypoints sequentially
        orig_keypoints.extend([[jx, jy], [cx, cy], [tx, ty]])

        orig_bboxes.append([row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']])
        orig_cats.append(0)

    for aug_idx in range(AUG_COPIES):
        # First copy is always the un-augmented original
        if aug_idx == 0:
            transformed = {'image': img, 'bboxes': orig_bboxes, 'keypoints': orig_keypoints, 'category_ids': orig_cats}
            out_img_name = img_name
        else:
            transformed = transform(image=img, bboxes=orig_bboxes, keypoints=orig_keypoints, category_ids=orig_cats)
            base_name, ext = os.path.splitext(img_name)
            out_img_name = f"{base_name}_aug{aug_idx}{ext}"

        t_img = transformed['image']
        t_bboxes = transformed['bboxes']
        t_keypoints = transformed['keypoints']

        # Save image
        cv2.imwrite(os.path.join(OUT_DIR, "images/train", out_img_name), t_img)

        # Write Label
        label_name = os.path.splitext(out_img_name)[0] + ".txt"
        with open(os.path.join(OUT_DIR, "labels/train", label_name), 'w') as f:

            for j, bbox in enumerate(t_bboxes):
                # Normalize Bbox
                x_min, y_min, bw, bh = bbox
                nx_c = (x_min + (bw / 2)) / img_w
                ny_c = (y_min + (bh / 2)) / img_h
                nw = bw / img_w
                nh = bh / img_h

                t_jx, t_jy = t_keypoints[j*3]      # Joint
                t_cx, t_cy = t_keypoints[j*3 + 1]  # Center
                t_tx, t_ty = t_keypoints[j*3 + 2]  # Tab


                line = (
                    f"0 {nx_c:.6f} {ny_c:.6f} {nw:.6f} {nh:.6f} "
                    f"{t_jx/img_w:.6f} {t_jy/img_h:.6f} 2 "
                    f"{t_cx/img_w:.6f} {t_cy/img_h:.6f} 2 "
                    f"{t_tx/img_w:.6f} {t_ty/img_h:.6f} 2\n"
                )
                f.write(line)

print("✅ Augmentation and Label Generation Complete!")

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_6162/525495752.py:24: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=45, p=0.8, border_mode=cv2.BORDER_CONSTANT, value=(0,0,0)),
/tmp/ipykernel_6162/525495752.py:26: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


Processing 70 images into 350 total samples...
✅ Augmentation and Label Generation Complete!


In [4]:
yaml_content = """
path: /content/tube_dataset
train: images/train
val: images/train  # For 350 images, validating on training set is standard practice

# 3 Keypoints: Joint, Center, Tab
kpt_shape: [3, 3]
names:
  0: tube
"""

with open('/content/pose_config.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ pose_config.yaml created successfully.")

✅ pose_config.yaml created successfully.


In [5]:
from ultralytics import YOLO

# Load the S-Model
model = YOLO('yolov8s-pose.pt')

print("🚀 Starting training with optimized hyperparameters...")

model.train(
    data='/content/pose_config.yaml',
    epochs=150,           # Small dataset needs more epochs
    imgsz=640,
    batch=8,
    name='tube_pose_s_v2',
    freeze=0,             # Unfreeze everything for full fine-tuning

    degrees=180.0,        # Tubes can be at any 360° orientation (CRUCIAL)
    scale=0.2,
    hsv_v=0.3,
    flipud=0.5,           # Vertical flip
    fliplr=0.5,           # Horizontal flip
    mosaic=0.0,

    # Optimizer settings
    patience=30,          # Early stopping
    lr0=0.001,
    lrf=0.01,
)

print("✅ Training complete. Best weights saved to runs/pose/tube_pose_s_v2/weights/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Starting training with optimized hyperparameters...
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pose_config.yaml, degrees=180.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=0,

In [9]:
import os
import math
import numpy as np
import pandas as pd
import cv2
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# ==========================================
# 1. CONFIGURATION
# ==========================================
POSE_MODEL_PATH = "/content/best_final.pt"

IMG_DIR = "/content/drive/MyDrive/zeon_sys_dataset/images"
CSV_PATH = "/content/drive/MyDrive/zeon_sys_dataset/annotations.csv"

MATCH_THRESHOLD = 30.0

print("Loading S-Model v2 (Pure Keypoints) and data...")
pose_model = YOLO(POSE_MODEL_PATH)
df = pd.read_csv(CSV_PATH)

# ==========================================
# 2. CORE LOGIC
# ==========================================
def get_angle_from_pose(kpts):
    jx, jy = kpts[0] # Joint
    tx, ty = kpts[2] # Tab
    dx = tx - jx
    dy = jy - ty
    angle_rad = math.atan2(dy, dx)
    return (math.degrees(angle_rad) + 360) % 360

def run_pure_pose_inference(img_path):
    results = pose_model(img_path, augment=True, verbose=False)[0]
    predictions = []

    if results.keypoints is not None and results.keypoints.xy is not None:
        kpts_array = results.keypoints.xy.cpu().numpy()
        for kpts in kpts_array:
            if len(kpts) == 3:
                center_x, center_y = kpts[1]
                angle = get_angle_from_pose(kpts)
                predictions.append([center_x, center_y, angle])
    return predictions

def get_angle_error(pred_deg, gt_deg):
    diff = abs(pred_deg - gt_deg)
    return min(diff, 360.0 - diff)

def evaluate_predictions(preds, gts):
    if len(preds) == 0: return 0, 0, len(gts), []

    preds_arr, gts_arr = np.array(preds), np.array(gts)
    dist_matrix = cdist(preds_arr[:, :2], gts_arr[:, :2])
    row_ind, col_ind = linear_sum_assignment(dist_matrix)

    tp, fp, fn = 0, 0, 0
    angle_errors = []
    matched_preds, matched_gts = set(), set()

    for r, c in zip(row_ind, col_ind):
        if dist_matrix[r, c] <= MATCH_THRESHOLD:
            tp += 1
            matched_preds.add(r)
            matched_gts.add(c)
            err = get_angle_error(preds_arr[r, 2], gts_arr[c, 2])
            angle_errors.append(err)

    fp = len(preds) - len(matched_preds)
    fn = len(gts) - len(matched_gts)
    return tp, fp, fn, angle_errors

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
unique_images = df['image'].unique()
print(f"Running inference on {len(unique_images)} images...\n")

total_tp, total_fp, total_fn = 0, 0, 0
all_angle_errors = []
missing_images = 0

for img_name in unique_images:
    gt_rows = df[df['image'] == img_name]
    gts = gt_rows[['center_x', 'center_y', 'angle_deg']].values.tolist()

    img_path = os.path.join(IMG_DIR, img_name)

    if not os.path.exists(img_path):
        missing_images += 1
        total_fn += len(gts)
        continue

    preds = run_pure_pose_inference(img_path)

    tp, fp, fn, errs = evaluate_predictions(preds, gts)
    total_tp += tp
    total_fp += fp
    total_fn += fn
    all_angle_errors.extend(errs)

if missing_images > 0:
    print(f"⚠️ WARNING: {missing_images} images from the CSV were NOT found in the folder!\n")

# ==========================================
# 4. FINAL REPORT
# ==========================================
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
mae = np.mean(all_angle_errors) if all_angle_errors else 0

print(f"--- YOLOv8s-Pose v2 (Pure Keypoints + TTA) RESULTS ---")
print(f"Centers Detected: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")
print(f"Angle Mean Abs Error (MAE): {mae:.2f} degrees")
print(f"(TP: {total_tp}, FP: {total_fp}, FN: {total_fn})")

Loading S-Model v2 (Pure Keypoints) and data...
Running inference on 70 images...

WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'au